# Semantic search sur le catalogue OMD

Recherche en langage naturel sur les metadonnees OMD (tables, colonnes, descriptions,
tags, glossaire), sans Elasticsearch : embeddings locaux + produit scalaire numpy.

Le code vit dans `src/retrieval/` ; ce notebook le pilote de bout en bout et mesure
l'effet des metadonnees sur la pertinence.

Modele : `paraphrase-multilingual-MiniLM-L12-v2` (220 Mo, telecharge une fois,
puis 100 % local — aucune metadonnee ne sort de la machine).

In [ ]:
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from src.retrieval.catalog import GlossaryEntry, load_catalog  # noqa: E402
from src.retrieval.document import build_documents  # noqa: E402
from src.retrieval.index import SemanticIndex  # noqa: E402

SERVICE = "banking db"
SCHEMAS = {"ref", "stg", "ods", "dmt", "tec"}  # schemas metier, hors schemas systeme Oracle

## 1. Lire le catalogue OMD

`load_catalog` fait des GET uniquement (SDK OMD, pagination geree) : tables, colonnes,
descriptions, tags et glossaire.

In [ ]:
catalog = load_catalog(SERVICE, SCHEMAS)


def stats(catalog):
    colonnes = [c for t in catalog.tables for c in (t.columns or [])]
    return {
        "tables": len(catalog.tables),
        "colonnes": len(colonnes),
        "tables decrites": sum(1 for t in catalog.tables if t.description),
        "colonnes decrites": sum(1 for c in colonnes if c.description),
        "colonnes taguees": sum(1 for c in colonnes if c.tags),
        "termes de glossaire": len(catalog.glossary),
    }


stats(catalog)

## 2. Composer les documents indexables

Chaque document porte deux textes embeddes separement — l'**identite** (identifiants
deplies) et le **sens** (description + glossaire) — et des facettes structurees (schema,
type, tags de classification) qui servent de filtres et ne sont jamais embeddees.

Deux vecteurs et pas un seul parce qu'un vecteur unique moyenne : ajouter une
description a une table bien nommee ferait alors baisser son score. Le score final est
le `max` des deux similarites (voir `index.py`).

In [ ]:
documents = build_documents(catalog)
print(f"{len(documents)} documents\n")

exemple = next(d for d in documents if d.name == "DMT_CPT_MVT_J")
print("identite :", exemple.identity_text)
print("sens     :", exemple.meaning_text or "(rien de documente)")
print("facettes :", exemple.schema, exemple.entity_type, exemple.tags)

## 3. Indexer et chercher

Vecteurs normes empiles dans une matrice `(N, 384)` : le cosinus se reduit a un
`matmul`. A ~10^3 documents la recherche est exacte et instantanee.

In [ ]:
debut = time.perf_counter()
index = SemanticIndex.build(documents)
index.save()
documentes = sum(1 for d in documents if d.has_meaning)
print(f"{index.embeddings.shape} en {time.perf_counter() - debut:.1f}s "
      f"({documentes} document(s) avec du sens) -> build/retrieval/")

In [ ]:
def montre(question, index, k=3, **filtres):
    print(f"\nQ: {question}")
    for hit in index.search(question, top_k=k, **filtres):
        doc = hit.document
        marque = " +desc" if doc.has_description else ""
        print(f"   {hit.score:.3f}  {doc.entity_type:6} {doc.name}{marque}")


QUESTIONS = [
    "solde journalier d'un compte",
    "mouvements debiteurs du mois",
    "capital restant du sur un credit",
    "adresse du client",
]

for question in QUESTIONS:
    montre(question, index, entity_type="table")

## 4. Filtrer

Les facettes s'appliquent en masque sur les scores : schema, type d'entite, tags de
classification, presence d'une description.

In [ ]:
montre("solde d'un compte", index, entity_type="column", schema="dmt")
montre("solde d'un compte", index, entity_type="table", described_only=True)

## 5. Ce que changent les metadonnees

Le catalogue est aujourd'hui vide de sens (0 description, 0 tag, 0 terme). Sur la vraie
base il ne le sera pas. On simule ici l'etat cible **en memoire** (aucune ecriture vers
OMD) pour mesurer l'ecart : quelques descriptions, un glossaire avec synonymes, un tag
de glossaire sur une colonne.

In [ ]:
from metadata.generated.schema.type.basic import Markdown
from metadata.generated.schema.type.tagLabel import LabelType, State, TagLabel, TagSource

DESCRIPTIONS_TABLE = {
    "DMT_CPT_MVT_J": "Mouvements comptables journaliers par compte : une ligne par "
                     "operation debitrice ou creditrice, montant en devise d'origine et en CHF.",
    "DMT_CPT_SLD_J": "Solde de fin de journee par compte, en francs suisses.",
    "ODS_D_CLI_ADR": "Adresses postales des clients, historisees par periode de validite.",
    "DMT_F_CRD_ENC_M": "Encours de credit mensuels par dossier et par agence.",
    "v_ref_fin_txc_chf": "Cours de conversion quotidiens des devises vers le franc suisse.",
}
DESCRIPTIONS_COLONNE = {
    ("DMT_CPT_MVT_J", "mt_mvt"): "Montant de l'operation dans la devise d'origine.",
    ("DMT_CPT_MVT_J", "cd_typ_ope"): "Code du type d'operation : debit, credit, virement, "
                                     "prelevement, retrait.",
    ("ODS_D_CLI_ADR", "rue"): "Libelle de voie de l'adresse postale du client.",
    ("DMT_F_CRD_ENC_M", "mt_crd_restant"): "Capital restant du sur le dossier de credit "
                                           "a la fin du mois.",
}
GLOSSAIRE = [
    GlossaryEntry("Banque.Solde disponible", "Solde disponible",
                  "Montant utilisable immediatement par le client.",
                  ["SLD", "avoir disponible"]),
    GlossaryEntry("Banque.Mouvement", "Mouvement",
                  "Ecriture comptable passee sur un compte.",
                  ["MVT", "operation", "transaction"]),
    GlossaryEntry("Banque.Encours", "Encours",
                  "Capital restant du sur un credit a une date donnee.",
                  ["ENC"]),
    GlossaryEntry("Banque.Taux de change", "Taux de change",
                  "Cours de conversion d'une devise vers une autre.",
                  ["TXC", "TX"]),
]


def enrichir_en_memoire(catalog):
    """Simule l'etat du catalogue une fois enrichi (ce que sink/omd.py ecrira)."""
    catalog.glossary.update({e.fqn: e for e in GLOSSAIRE})
    for table in catalog.tables:
        if table.name.root in DESCRIPTIONS_TABLE:
            table.description = Markdown(DESCRIPTIONS_TABLE[table.name.root])
        for colonne in table.columns or []:
            cle = (table.name.root, colonne.name.root)
            if cle in DESCRIPTIONS_COLONNE:
                colonne.description = Markdown(DESCRIPTIONS_COLONNE[cle])
            # Le tag de glossaire fait entrer la definition du terme dans la fiche.
            if cle == ("DMT_F_CRD_ENC_M", "mt_crd_restant"):
                colonne.tags = [TagLabel(tagFQN="Banque.Encours", source=TagSource.Glossary,
                                         labelType=LabelType.Manual, state=State.Suggested)]
    return catalog


catalog_enrichi = enrichir_en_memoire(load_catalog(SERVICE, SCHEMAS))
documents_enrichis = build_documents(catalog_enrichi)
index_enrichi = SemanticIndex.build(documents_enrichis)

stats(catalog_enrichi)

In [ ]:
enrichi = next(d for d in documents_enrichis if d.name == "DMT_CPT_MVT_J")
print("identite :", enrichi.identity_text)
print("sens     :", enrichi.meaning_text)

## 6. Mesurer, pas regarder

Cinq questions, la table attendue pour chacune : le minimum pour comparer deux
configurations autrement qu'a l'oeil. Un vrai golden set fait ~50 entrees.

In [ ]:
GOLDEN = {
    "solde journalier d'un compte": "DMT_CPT_SLD_J",
    "mouvements debiteurs du mois": "DMT_CPT_MVT_J",
    "capital restant du sur un credit": "DMT_F_CRD_ENC_M",
    "adresse du client": "ODS_D_CLI_ADR",
    "taux de change vers le franc suisse": "v_ref_fin_txc_chf",
}


def recall_at_k(index, k=3):
    trouves = []
    for question, attendu in GOLDEN.items():
        noms = [h.document.name for h in index.search(question, top_k=k, entity_type="table")]
        rang = noms.index(attendu) + 1 if attendu in noms else None
        trouves.append(rang is not None)
        etat = f"rang {rang}" if rang else "RATE"
        print(f"   {question:38} {etat:8} top1={noms[0]}")
    print(f"   recall@{k} = {sum(trouves)}/{len(GOLDEN)}\n")


print("-- catalogue actuel (identifiants seuls)")
recall_at_k(index)
print("-- catalogue enrichi (identifiants + descriptions + glossaire)")
recall_at_k(index_enrichi)

## 6. A retenir

- Le branchement est acquis : `catalog.py` lit deja descriptions, tags et glossaire.
  Chaque description ecrite par `sink/omd.py` ameliore la recherche au run suivant,
  sans toucher au code.
- Les descriptions font gagner des questions que les identifiants seuls ratent
  (« capital restant du » -> `DMT_F_CRD_ENC_M`, introuvable autrement : ces trois mots
  n'apparaissent dans aucun identifiant de la table).
- La fusion des deux canaux a ete **calee sur ce golden set**, pas choisie a priori :
  `max(identite, melange)` donne 4/5, le melange seul 3/5, le max des canaux bruts 3/5
  (voir le docstring de `SemanticIndex`). A refaire quand le catalogue sera
  majoritairement documente : ici 5 tables sur 101 le sont, regime tres asymetrique.
- Echec restant : « mouvements debiteurs du mois » remonte `ODS_F_CPT_MVT_BCK_2019`,
  une table de sauvegarde. Les `_BCK_*` et `_OLD` polluent le haut du classement et
  meritent un filtre (par nom, ou par un tag de classification pose dans OMD - c'est
  exactement le genre de decision qui a sa place dans le catalogue, pas dans le code).